# Experiment B (alpha-fixed): Regularised GLM with L2-penalised Team Dummies — alpha floor 0.1

Identical to notebook 15 (Experiment B) except the inner-CV alpha grid is raised to
[0.1, 0.5, 1.0, 5.0, 10.0], enforcing a minimum alpha floor of 0.1.

**Motivation**: In Experiment B the inner CV selected alpha=0.001 on the 2014/2018/2022
training sets, allowing team dummies to overfit. The 2014 fold regressed −0.281 pts vs the GLM
baseline, dragging the pooled mean down. Removing near-zero alphas forces the model to use
meaningful shrinkage in every fold.

**Protocol**:
- Same 4-fold LOTO-CV as all prior experiments (WC 2010/2014/2018/2022 as held-out)
- Grid search over alpha in [0.1, 0.5, 1.0, 5.0, 10.0] using inner 3-fold CV on the training set
- Unseen teams in eval (only Brazil in 2010 fold): handled with `handle_unknown='ignore'`
- Baseline comparison: Poisson GLM covariate-only (4.336 pts/match)

In [1]:
import sys
sys.path.insert(0, '../../src')

import json
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from functools import lru_cache
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold
from pathlib import Path

from evaluation.sporza import score_breakdown, sporza_points_series

DATA = Path('../../data/processed')
WC_YEARS = [2010, 2014, 2018, 2022]
MAX_GOALS = 8

# Continuous features (same as GLM baseline)
CONT_FEATURES = [
    'home_elo', 'away_elo', 'elo_diff', 'is_neutral',
    'home_form_wr', 'away_form_wr', 'home_form_gf', 'away_form_gf',
    'home_form_ga', 'away_form_ga', 'tournament_weight'
]
# Categorical features: team identity
CAT_FEATURES_HOME = ['home_team', 'away_team']
CAT_FEATURES_AWAY = ['away_team', 'home_team']  # perspective-flipped for away model

# For away-perspective rows, we use mirrored continuous features
CONT_FEATURES_AWAY = [
    'away_elo', 'home_elo', 'elo_diff', 'is_neutral',
    'away_form_wr', 'home_form_wr', 'away_form_gf', 'home_form_gf',
    'away_form_ga', 'home_form_ga', 'tournament_weight'
]

# Alpha floor of 0.1 — prevents near-zero regularisation overfitting seen in Experiment B
ALPHA_GRID = [0.1, 0.5, 1.0, 5.0, 10.0]

In [2]:
def bootstrap_ci(pts: pd.Series, n_boot: int = 10000) -> dict:
    rng = np.random.default_rng(42)
    vals = pts.values
    boot = np.array([rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {'mean': float(vals.mean()), 'ci_lo': float(lo), 'ci_hi': float(hi)}


@lru_cache(maxsize=50000)
def best_score_cached(lh_r: float, la_r: float) -> tuple:
    """Score (h, a) maximising expected Sporza pts given Poisson(lh) x Poisson(la)."""
    goals = np.arange(MAX_GOALS + 1)
    ph_pmf = poisson.pmf(goals, lh_r)
    pa_pmf = poisson.pmf(goals, la_r)
    joint = np.outer(ph_pmf, pa_pmf)

    def sporza_mat(pred_h: int, pred_a: int) -> float:
        pts = 0.0
        for ah in range(MAX_GOALS + 1):
            for aa in range(MAX_GOALS + 1):
                p = joint[ah, aa]
                if p < 1e-9:
                    continue
                if pred_h == ah and pred_a == aa:
                    pts += p * 10
                elif (pred_h - pred_a) == (ah - aa):
                    pts += p * 7
                elif np.sign(pred_h - pred_a) == np.sign(ah - aa):
                    pts += p * 5
                else:
                    pts += p * 1
        return pts

    best_pts, best_h, best_a = -1.0, 1, 0
    for ph_idx in range(MAX_GOALS + 1):
        for pa_idx in range(MAX_GOALS + 1):
            ep = sporza_mat(ph_idx, pa_idx)
            if ep > best_pts:
                best_pts, best_h, best_a = ep, ph_idx, pa_idx
    return best_h, best_a

In [3]:
def build_preprocessor(train_teams: list[str]) -> ColumnTransformer:
    """Build a ColumnTransformer that scales continuous features and one-hot encodes team columns.

    handle_unknown='ignore' silently zeros out unseen team dummies (e.g. Brazil in 2010 fold).
    categories are fixed to training teams to prevent eval leakage.
    """
    team_enc = OneHotEncoder(
        categories=[train_teams, train_teams],
        handle_unknown='ignore',
        sparse_output=False,
    )
    return ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), list(range(len(CONT_FEATURES)))),
            ('teams', team_enc, [len(CONT_FEATURES), len(CONT_FEATURES) + 1]),
        ]
    )


def build_stacked_X(
    df: pd.DataFrame, cont_home: list, cont_away: list, cat_home: list, cat_away: list
) -> np.ndarray:
    """Stack home and away rows into a single feature matrix for the shared Poisson regressor.

    Each match contributes two rows: home perspective (predict home goals) and
    away perspective (predict away goals). Columns: [cont_features..., team1, team2].
    """
    X_h = np.column_stack(
        [df[cont_home].fillna(df[cont_home].median()).values, df[cat_home].values]
    )
    X_a = np.column_stack(
        [df[cont_away].fillna(df[cont_away].median()).values, df[cat_away].values]
    )
    return np.vstack([X_h, X_a])


def build_eval_X(
    df: pd.DataFrame, cont_feats: list, cat_feats: list
) -> np.ndarray:
    return np.column_stack(
        [df[cont_feats].fillna(df[cont_feats].median()).values, df[cat_feats].values]
    )

In [4]:
manifest = json.loads((DATA / 'split_manifest.json').read_text())
autofill_mean = manifest['autofill_baseline']['pooled_mean_pts']
print(f'Autofill baseline: {autofill_mean:.3f} pts/match')
print(f'GLM baseline:      4.336 pts/match')

Autofill baseline: 4.137 pts/match
GLM baseline:      4.336 pts/match


## Inner alpha cross-validation

For each LOTO fold, we run a 3-fold inner CV on the *training* split to pick the best alpha.
The inner CV uses mean Poisson deviance (scikit-learn's negative Poisson deviance scorer) as a
proxy — we do NOT use Sporza pts in the inner loop to avoid leaking eval information.

Note: this means we're optimising log-likelihood, not Sporza score directly. A mismatch between
the two is possible but acceptable — the inner CV is only choosing regularisation strength.

In [5]:
all_pts = []
fold_results = []
fold_best_alphas = {}

for year in WC_YEARS:
    train = pd.read_parquet(DATA / f'train_fold_{year}.parquet')
    evalf = pd.read_parquet(DATA / f'eval_fold_{year}.parquet')

    # Collect all teams seen in training (for fixed OHE categories)
    train_teams = sorted(
        set(train['home_team'].unique()) | set(train['away_team'].unique())
    )

    # Build stacked training matrix: each match → 2 rows
    X_tr = build_stacked_X(train, CONT_FEATURES, CONT_FEATURES_AWAY, CAT_FEATURES_HOME, CAT_FEATURES_AWAY)
    y_tr = np.concatenate([train['home_score'].values, train['away_score'].values])

    # Inner 3-fold CV to select best alpha
    preprocessor = build_preprocessor(train_teams)
    inner_pipe = Pipeline([
        ('pre', preprocessor),
        ('poisson', PoissonRegressor(max_iter=500)),
    ])
    gs = GridSearchCV(
        inner_pipe,
        param_grid={'poisson__alpha': ALPHA_GRID},
        cv=KFold(n_splits=3, shuffle=True, random_state=42),
        scoring='neg_mean_poisson_deviance',
        refit=True,
        n_jobs=-1,
    )
    gs.fit(X_tr, y_tr)
    best_alpha = gs.best_params_['poisson__alpha']
    fold_best_alphas[year] = best_alpha
    print(f'WC {year}: best_alpha={best_alpha}  inner_cv_score={gs.best_score_:.4f}')

    # Predict on eval fold using the refitted model
    best_pipe = gs.best_estimator_

    X_ev_home = build_eval_X(evalf, CONT_FEATURES, CAT_FEATURES_HOME)
    X_ev_away = build_eval_X(evalf, CONT_FEATURES_AWAY, CAT_FEATURES_AWAY)
    lh = best_pipe.predict(X_ev_home)
    la = best_pipe.predict(X_ev_away)

    preds = [best_score_cached(round(float(h), 2), round(float(a), 2)) for h, a in zip(lh, la)]
    pred_home = pd.Series([p[0] for p in preds])
    pred_away = pd.Series([p[1] for p in preds])

    bd = score_breakdown(
        pred_home, pred_away,
        evalf['home_score'].reset_index(drop=True),
        evalf['away_score'].reset_index(drop=True),
    )
    pts = sporza_points_series(
        pred_home, pred_away,
        evalf['home_score'].reset_index(drop=True),
        evalf['away_score'].reset_index(drop=True),
    )
    all_pts.extend(pts.tolist())
    fold_results.append({
        'year': year,
        'mean_pts': bd['mean_pts'],
        'pct_exact': bd['pct_exact'],
        'pct_correct_result': bd['pct_correct_result'],
        'best_alpha': best_alpha,
    })
    print(f'       mean_pts={bd["mean_pts"]:.3f}  exact={bd["pct_exact"]:.1%}  correct={bd["pct_correct_result"]:.1%}')
    print()

WC 2010: best_alpha=0.1  inner_cv_score=-1.2712
       mean_pts=4.312  exact=18.8%  correct=53.1%



WC 2014: best_alpha=0.1  inner_cv_score=-1.2857
       mean_pts=4.734  exact=17.2%  correct=62.5%



WC 2018: best_alpha=0.1  inner_cv_score=-1.2560
       mean_pts=4.562  exact=18.8%  correct=57.8%



WC 2022: best_alpha=0.1  inner_cv_score=-1.2417
       mean_pts=3.766  exact=7.8%  correct=54.7%



In [6]:
pooled_pts = pd.Series(all_pts)
ci = bootstrap_ci(pooled_pts)

print(f'Pooled LOTO-CV ({len(pooled_pts)} matches):')
print(f'  Reg GLM + team dummies (alpha-fixed):  {ci["mean"]:.3f} pts/match  95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]')
print(f'  Poisson GLM baseline:                  4.336 pts/match')
print(f'  Experiment B (alpha unrestricted):     4.379 pts/match')
print(f'  Autofill baseline:                     {autofill_mean:.3f} pts/match')
print(f'  Delta vs GLM baseline:                 {ci["mean"] - 4.336:+.3f} pts/match')
print(f'  Delta vs Experiment B:                 {ci["mean"] - 4.379:+.3f} pts/match')
print(f'  Delta vs autofill:                     {ci["mean"] - autofill_mean:+.3f} pts/match')

beats_glm = ci['ci_lo'] > 4.336
beats_autofill = ci['ci_lo'] > autofill_mean
print(f'  CI lower > GLM baseline: {beats_glm} → {"statistically beats GLM" if beats_glm else "not conclusive vs GLM"}')
print(f'  CI lower > autofill:     {beats_autofill} → {"statistically beats autofill" if beats_autofill else "not conclusive vs autofill"}')

Pooled LOTO-CV (256 matches):
  Reg GLM + team dummies (alpha-fixed):  4.344 pts/match  95% CI [3.945, 4.750]
  Poisson GLM baseline:                  4.336 pts/match
  Experiment B (alpha unrestricted):     4.379 pts/match
  Autofill baseline:                     4.137 pts/match
  Delta vs GLM baseline:                 +0.008 pts/match
  Delta vs Experiment B:                 -0.035 pts/match
  Delta vs autofill:                     +0.207 pts/match
  CI lower > GLM baseline: False → not conclusive vs GLM
  CI lower > autofill:     False → not conclusive vs autofill


In [7]:
print('Per-fold results:')
print(f'{"Fold":<8} {"mean_pts":>10} {"exact":>8} {"correct":>10} {"best_alpha":>12}')
glm_fold = {2010: 4.141, 2014: 4.875, 2018: 4.562, 2022: 3.766}
exp_b_fold = {2010: 4.312, 2014: 4.594, 2018: 4.750, 2022: 3.859}
for r in fold_results:
    delta_glm = r['mean_pts'] - glm_fold[r['year']]
    delta_b = r['mean_pts'] - exp_b_fold[r['year']]
    print(f'WC {r["year"]}   {r["mean_pts"]:>10.3f} {r["pct_exact"]:>8.1%} {r["pct_correct_result"]:>10.1%} {r["best_alpha"]:>12}  (Δ vs GLM: {delta_glm:+.3f}, Δ vs B: {delta_b:+.3f})')

Per-fold results:
Fold       mean_pts    exact    correct   best_alpha
WC 2010        4.312    18.8%      53.1%          0.1  (Δ vs GLM: +0.171, Δ vs B: +0.000)
WC 2014        4.734    17.2%      62.5%          0.1  (Δ vs GLM: -0.141, Δ vs B: +0.140)
WC 2018        4.562    18.8%      57.8%          0.1  (Δ vs GLM: +0.000, Δ vs B: -0.188)
WC 2022        3.766     7.8%      54.7%          0.1  (Δ vs GLM: -0.000, Δ vs B: -0.093)


In [8]:
mlflow.set_tracking_uri('sqlite:////Users/seppe.vanswegenoven/projects/wk_2026_match_predictor/mlflow.db')
mlflow.set_experiment('wk2026-tabular-2026-06-14')

with mlflow.start_run(run_name='reg_glm_team_dummies_alphafixed_loto'):
    mlflow.set_tags({
        'modality': 'tabular',
        'approach': 'regularised_glm_team_dummies',
        'eval_strategy': 'loto_cv',
        'dataset_version': 'split_v2',
        'experiment': 'B_alphafixed',
    })
    mlflow.log_params({
        'cont_features': ','.join(CONT_FEATURES),
        'team_features': 'home_team,away_team (OHE)',
        'alpha_grid': str(ALPHA_GRID),
        'inner_cv_folds': 3,
        'inner_cv_scorer': 'neg_mean_poisson_deviance',
        'score_strategy': 'max_expected_sporza_pts',
        'max_goals_grid': MAX_GOALS,
        'handle_unknown': 'ignore',
    })
    mlflow.log_metric('loto_mean_sporza_pts', ci['mean'])
    mlflow.log_metric('loto_ci_lo', ci['ci_lo'])
    mlflow.log_metric('loto_ci_hi', ci['ci_hi'])
    mlflow.log_metric('autofill_baseline', autofill_mean)
    mlflow.log_metric('delta_vs_autofill', ci['mean'] - autofill_mean)
    mlflow.log_metric('delta_vs_glm_baseline', ci['mean'] - 4.336)
    mlflow.log_metric('delta_vs_exp_b', ci['mean'] - 4.379)
    for r in fold_results:
        mlflow.log_metric(f'fold_{r["year"]}_mean_pts', r['mean_pts'])
        mlflow.log_metric(f'fold_{r["year"]}_pct_exact', r['pct_exact'])
        mlflow.log_metric(f'fold_{r["year"]}_best_alpha', r['best_alpha'])
    run_id = mlflow.active_run().info.run_id

print(f'MLflow run_id: {run_id}')

MLflow run_id: 7c0aaaee0b824205b6497ca86e21d8c0


## Interpretation

Three possible outcomes and their implications:

**A) Alpha-fixed > Experiment B (> 4.379 pts):**
The near-zero alpha was genuinely overfitting — forcing stronger shrinkage helps. The team-dummy
approach is viable with a proper floor. Next: finer HPO around the winning alpha range.

**B) Alpha-fixed ≈ Experiment B (within ±0.02 pts):**
The overfitting was concentrated in folds where the unrestricted grid happened to pick a low alpha;
with the floor the model behaves similarly. Delta vs GLM is unchanged — team dummies give marginal
benefit. Next: Experiment C (pi-ratings) for a different angle.

**C) Alpha-fixed < Experiment B:**
Surprising — the low-alpha folds were actually benefiting from near-zero regularisation (low bias
on large training sets). Forcing the floor hurts those folds. Suggests the overfitting diagnosis
was fold-specific, not systematic.